In [2]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, log_loss, mean_squared_error
from torch.utils.data import DataLoader
from tqdm import tqdm
from itertools import combinations
from sklearn.model_selection import train_test_split

import heapq
from random import randrange
from random import seed as set_seed
import numpy as np
#from numba import njit, prange
from pandas.api.types import is_numeric_dtype

import numpy as np
import pandas as pd
import torch.utils.data

from scipy.sparse.linalg import svds

In [3]:
import pandas as pd
import gzip

def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

def getDF(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

df = getDF('reviews_Beauty_5.json.gz')

In [4]:
df.head(10)

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A1YJEY40YUW4SE,7806397051,Andrea,"[3, 4]",Very oily and creamy. Not at all what I expect...,1.0,Don't waste your money,1391040000,"01 30, 2014"
1,A60XNB876KYML,7806397051,Jessica H.,"[1, 1]",This palette was a decent price and I was look...,3.0,OK Palette!,1397779200,"04 18, 2014"
2,A3G6XNM240RMWA,7806397051,Karen,"[0, 1]",The texture of this concealer pallet is fantas...,4.0,great quality,1378425600,"09 6, 2013"
3,A1PQFP6SAJ6D80,7806397051,Norah,"[2, 2]",I really can't tell what exactly this thing is...,2.0,Do not work on my face,1386460800,"12 8, 2013"
4,A38FVHZTNQ271F,7806397051,Nova Amor,"[0, 0]","It was a little smaller than I expected, but t...",3.0,It's okay.,1382140800,"10 19, 2013"
5,A3BTN14HIZET6Z,7806397051,"S. M. Randall ""WildHorseWoman""","[1, 2]","I was very happy to get this palette, now I wi...",5.0,Very nice palette!,1365984000,"04 15, 2013"
6,A1Z59RFKN0M5QL,7806397051,"tasha ""luvely12b""","[1, 3]",PLEASE DONT DO IT! this just rachett the palet...,1.0,smh!!!,1376611200,"08 16, 2013"
7,AWUO9P6PL1SY8,7806397051,TreMagnifique,"[0, 1]","Chalky,Not Pigmented,Wears off easily,Not a Co...",2.0,"Chalky, Not Pigmented, Wears off easily, Not a...",1378252800,"09 4, 2013"
8,A3LMILRM9OC3SA,9759091062,NaN,"[0, 0]",Did nothing for me. Stings when I put it on. I...,2.0,"no Lightening, no Brightening,......NOTHING",1405209600,"07 13, 2014"
9,A30IP88QK3YUIO,9759091062,Amina Bint Ibraheem,"[0, 0]",I bought this product to get rid of the dark s...,3.0,Its alright,1388102400,"12 27, 2013"


In [5]:
new_df = df[['reviewerID', 'asin', 'overall', 'unixReviewTime']].copy()
new_df.columns = ['user_id', 'item_id', 'rating', 'timestamp']
new_df['rating'] = 1
new_df.head(5)

,user_id,item_id,rating,timestamp
0,A1YJEY40YUW4SE,7806397051,1,1391040000
1,A60XNB876KYML,7806397051,1,1397779200
2,A3G6XNM240RMWA,7806397051,1,1378425600
3,A1PQFP6SAJ6D80,7806397051,1,1386460800
4,A38FVHZTNQ271F,7806397051,1,1382140800


In [6]:
new_df['user_id'], unique_user_ids = pd.factorize(new_df['user_id'])

new_df['item_id'], unique_item_ids = pd.factorize(new_df['item_id'])
new_df['user_id'] += 1
new_df['item_id'] += 1
new_df.head(5)

,user_id,item_id,rating,timestamp
0,1,1,1,1391040000
1,2,1,1,1397779200
2,3,1,1,1378425600
3,4,1,1,1386460800
4,5,1,1,1382140800


In [7]:
df_sorted = new_df.sort_values(by='timestamp')

test_treshold = int(len(df_sorted) * 0.98)
val_treshold = int(len(df_sorted) * 0.96)

train_val = df_sorted.head(test_treshold)
warm_test = df_sorted.tail(len(df_sorted) - test_treshold)
test = warm_test.loc[warm_test.groupby('user_id')['timestamp'].idxmax()]
warm_t = warm_test[~warm_test.index.isin(test.index)]


train = df_sorted.head(val_treshold)
test_val = df_sorted.tail(len(df_sorted) - val_treshold)
warm_val = test_val[~test_val.index.isin(warm_test.index)]
val = warm_val.loc[warm_val.groupby('user_id')['timestamp'].idxmax()]
warm_v = warm_val[~warm_val.index.isin(val.index)]


train = pd.concat([train, warm_v])

In [8]:
print("Train len: ", len(train))
print("Train users: ", len(train["user_id"].unique()))
print("Val len: ", len(val))
print("Val users: ", len(val["user_id"].unique()))
print("Test len: ", len(test))
print("Test users: ", len(test["user_id"].unique()))
print("Warm val len: ", len(warm_v))
print("Warm val users: ", len(warm_v["user_id"].unique()))
print("Warm test len: ", len(warm_t))
print("Warm test users: ", len(warm_t["user_id"].unique()))
print("test_val intersection users: ", np.intersect1d(val["user_id"].unique(), test["user_id"].unique()).shape[0])

Train len:  192801
Train users:  22235
Val len:  1730
Val users:  1730
Test len:  1670
Test users:  1670
Warm val len:  2240
Warm val users:  800
Warm test len:  2301
Warm test users:  771
test_val intersection users:  432


In [9]:
user_items = train.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/Beauty/train.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [16]:
train_warm_v = pd.concat([train, warm_v], ignore_index=True)
user_items = warm_v.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/Beauty/warm_val.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [21]:
filtered_train = train_warm_v[train_warm_v['user_id'].isin(val['user_id'])]
train_val = pd.concat([filtered_train, val], ignore_index=True)

user_items = train_val.groupby('user_id')['item_id'].apply(list).to_dict()
with open('data2/Beauty/val.txt', 'w') as f:
    for user_id, items in user_items.items():
        if len(items) > 1:
            f.write(f"{user_id} {' '.join(map(str, items))}\n")


In [18]:
unique_indices = val["user_id"].unique()
with open('data2/Beauty/val_users.txt', 'w') as f:
    for index in unique_indices:
        f.write(f"{index}\n")

# Чтение чисел из файла и сохранение их в список
with open('data2/Beauty/val_users.txt', 'r') as f:
    index_list = [int(line.strip()) for line in f]

In [19]:
filtered_train = df_sorted[df_sorted['user_id'].isin(test['user_id'])]


user_items = filtered_train.groupby('user_id')['item_id'].apply(list).to_dict()
with open('data2/Beauty/test.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")


In [20]:
user_items = df_sorted.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/Beauty/all_data.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")